## Players Data Load

In [ ]:
# import the libraries

import requests
import mysql.connector
from mysql.connector import Error

In [3]:
try:
    conn = mysql.connector.connect(
        host="localhost", database="nhl_analytics_hub", user="root", password="root"
    )
    if conn.is_connected:
        print("Connected successfully...")
    cur = conn.cursor()
    cur.execute("SELECT team_abbrev, team_id FROM teams")
    team_id_map = dict(cur.fetchall())
    print(team_id_map)
    for abbrev, team_id in team_id_map.items():
        url = f"https://api-web.nhle.com/v1/roster/{abbrev}/20252026"
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        roster = resp.json()

        all_players = (
            roster.get("forwards") + roster.get("defensemen") + roster.get("goalies")
        )

        for p in all_players:
            cur.execute(
                """
                    INSERT IGNORE INTO players (
                        player_id, team_id, first_name, last_name, position,
                        jersey_number, birth_date, birth_country, height_cm,
                        weight_kg, shoots_catches, headshot_url
                    )
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    p["id"],
                    team_id,
                    p["firstName"]["default"],
                    p["lastName"]["default"],
                    p.get("positionCode"),
                    p.get("sweaterNumber"),
                    p.get("birthDate"),
                    p.get("birthCountry"),
                    p.get("heightInCentimeters"),
                    p.get("weightInKilograms"),
                    p.get("shootsCatches"),
                    p.get("headshot"),
                ),
            )

    conn.commit()

except Error as e:
    print("Exception Occurred...")
    print(e)

finally:
    if conn.is_connected:
        cur.close()
        conn.close()

Connected successfully...
{'ANA': 20, 'BOS': 11, 'BUF': 19, 'CAR': 1, 'CBJ': 29, 'CGY': 21, 'CHI': 28, 'COL': 10, 'DAL': 8, 'DET': 26, 'EDM': 3, 'FLA': 13, 'LAK': 30, 'MIN': 2, 'MTL': 9, 'NJD': 6, 'NSH': 12, 'NYI': 15, 'NYR': 18, 'OTT': 4, 'PHI': 22, 'PIT': 31, 'SEA': 24, 'SJS': 23, 'STL': 16, 'TBL': 7, 'TOR': 27, 'UTA': 17, 'VAN': 32, 'VGK': 5, 'WPG': 14, 'WSH': 25}
